In [ ]:
import os
import random
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

from data_loader import load_windows, loso_splits

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
DL_PATH = r"C:\ML\StressDetection\DL"

HIDDEN_SIZE = 64
NUM_LAYERS  = 1
DROPOUT     = 0.3
BATCH_SIZE  = 128
LR          = 1e-3

# Early stopping
MAX_EPOCHS   = 30
PATIENCE     = 5
VAL_FRACTION = 0.10
MIN_DELTA    = 1e-4

print("Config loaded.")
print(f"  hidden={HIDDEN_SIZE}, layers={NUM_LAYERS}, dropout={DROPOUT}")
print(f"  batch={BATCH_SIZE}, lr={LR}")
print(f"  max_epochs={MAX_EPOCHS}, patience={PATIENCE}")

In [ ]:
X_raw, y, pids = load_windows(DL_PATH, file_type="valence")

In [ ]:
# =====================================
# VALENCE SMOOTHING — experiment 1
# =====================================
SMOOTH_WINDOW = 5   # 1 = baseline / no smoothing, 5 = moving average

def smooth_valence_windows(X_input, window=5):
    """
    Smooth each valence sequence independently.
    Shape in/out: (num_windows, 140, 1)
    """
    if window == 1:
        return X_input.copy()

    assert window % 2 == 1, "window must be odd: 3, 5, 7..."

    pad = window // 2
    kernel = np.ones(window, dtype=np.float32) / window
    X_smooth = np.empty_like(X_input, dtype=np.float32)

    for i in range(len(X_input)):
        seq = X_input[i, :, 0].astype(np.float32)

        # edge padding keeps the sequence length = 140
        padded = np.pad(seq, (pad, pad), mode="edge")
        X_smooth[i, :, 0] = np.convolve(padded, kernel, mode="valid")

    return X_smooth


X = smooth_valence_windows(X_raw, window=SMOOTH_WINDOW)

print("Smoothing window:", SMOOTH_WINDOW)
print("Input shape:", X.shape)
print("Raw first values:     ", X_raw[0, :8, 0])
print("Smoothed first values:", X[0, :8, 0])

In [ ]:
class ValenceLSTM(nn.Module):
    """
    Bidirectional LSTM + 4 summary statistics:
    mean, std, min, max
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        # BiLSTM output: forward + backward = hidden_size * 2
        # Plus 4 summary statistics
        self.fc = nn.Linear(hidden_size * 2 + 4, 2)

    def forward(self, x, stats):
        _, (h_n, _) = self.lstm(x)

        forward_hidden = h_n[-2]
        backward_hidden = h_n[-1]

        sequence_features = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        sequence_features = self.dropout(sequence_features)

        combined = torch.cat(
            (sequence_features, stats),
            dim=1
        )

        return self.fc(combined)


print("BiLSTM + stats model defined.")

In [ ]:
def set_fold_seed(held_out_pid):
    fold_seed = SEED + int(held_out_pid)

    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)


def extract_summary_stats(X_data):
    """
    Input shape: (N, 140, 1)
    Output per sequence: mean, std, min, max
    """
    seq = X_data[:, :, 0]

    stats = np.column_stack([
        seq.mean(axis=1),
        seq.std(axis=1),
        seq.min(axis=1),
        seq.max(axis=1)
    ])

    return stats.astype(np.float32)


def make_loader(X_data, stats_data, y_data, shuffle):
    X_tensor = torch.tensor(X_data, dtype=torch.float32)
    stats_tensor = torch.tensor(stats_data, dtype=torch.float32)
    y_tensor = torch.tensor(y_data, dtype=torch.long)

    return DataLoader(
        TensorDataset(X_tensor, stats_tensor, y_tensor),
        batch_size=BATCH_SIZE,
        shuffle=shuffle
    )


def split_train_validation_by_participant(X_train, y_train, train_pids, held_out_pid):
    """
    Validation participants are chosen only from LOSO training people.
    Held-out test participant is never used here.
    """
    unique_train_pids = np.unique(train_pids)

    n_val_pids = max(
        1,
        int(np.ceil(len(unique_train_pids) * VAL_FRACTION))
    )

    rng = np.random.default_rng(SEED + int(held_out_pid))

    val_pids = np.sort(
        rng.choice(
            unique_train_pids,
            size=n_val_pids,
            replace=False
        )
    )

    val_mask = np.isin(train_pids, val_pids)
    fit_mask = ~val_mask

    return (
        X_train[fit_mask],
        y_train[fit_mask],
        X_train[val_mask],
        y_train[val_mask],
        val_pids
    )


def normalize_sequences_and_stats(X_fit, X_val, X_test):
    """
    Fit normalization on fitting participants only.
    """
    # Normalize sequence
    seq_mean = X_fit.mean()
    seq_std = X_fit.std() + 1e-8

    X_fit_norm = (X_fit - seq_mean) / seq_std
    X_val_norm = (X_val - seq_mean) / seq_std
    X_test_norm = (X_test - seq_mean) / seq_std

    # Extract stats after sequence normalization
    stats_fit = extract_summary_stats(X_fit_norm)
    stats_val = extract_summary_stats(X_val_norm)
    stats_test = extract_summary_stats(X_test_norm)

    # Normalize stats using fitting set only
    stats_mean = stats_fit.mean(axis=0, keepdims=True)
    stats_std = stats_fit.std(axis=0, keepdims=True) + 1e-8

    stats_fit_norm = (stats_fit - stats_mean) / stats_std
    stats_val_norm = (stats_val - stats_mean) / stats_std
    stats_test_norm = (stats_test - stats_mean) / stats_std

    return (
        X_fit_norm, X_val_norm, X_test_norm,
        stats_fit_norm, stats_val_norm, stats_test_norm
    )


def train_one_fold(X_train_raw, y_train, train_pids,
                   X_test_raw, y_test, held_out_pid):

    set_fold_seed(held_out_pid)

    X_fit_raw, y_fit, X_val_raw, y_val, val_pids = \
        split_train_validation_by_participant(
            X_train_raw,
            y_train,
            train_pids,
            held_out_pid
        )

    (
        X_fit, X_val, X_test,
        stats_fit, stats_val, stats_test
    ) = normalize_sequences_and_stats(
        X_fit_raw,
        X_val_raw,
        X_test_raw
    )

    model = ValenceLSTM(
        input_size=1,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(device)

    counts = np.bincount(y_fit, minlength=2)

    class_weights = torch.tensor(
        [
            len(y_fit) / (2 * count) if count > 0 else 0.0
            for count in counts
        ],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    fit_loader = make_loader(X_fit, stats_fit, y_fit, shuffle=True)
    val_loader = make_loader(X_val, stats_val, y_val, shuffle=False)
    test_loader = make_loader(X_test, stats_test, y_test, shuffle=False)

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(MAX_EPOCHS):

        # Training
        model.train()

        for xb, sb, yb in fit_loader:
            xb = xb.to(device)
            sb = sb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb, sb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

        # Validation
        model.eval()

        total_val_loss = 0.0
        total_val_samples = 0

        with torch.no_grad():
            for xb, sb, yb in val_loader:
                xb = xb.to(device)
                sb = sb.to(device)
                yb = yb.to(device)

                logits = model(xb, sb)
                loss = criterion(logits, yb)

                total_val_loss += loss.item() * len(yb)
                total_val_samples += len(yb)

        val_loss = total_val_loss / total_val_samples

        # Early stopping
        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= PATIENCE:
                break

    # Restore best model
    model.load_state_dict(best_state)

    # Final test on held-out participant
    model.eval()

    all_preds = []
    all_probs = []

    with torch.no_grad():
        for xb, sb, yb in test_loader:
            xb = xb.to(device)
            sb = sb.to(device)

            logits = model(xb, sb)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    info = {
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "val_pids": val_pids.tolist()
    }

    return np.array(all_preds), np.array(all_probs), info


print("Helpers with BiLSTM + stats + early stopping defined.")

In [ ]:
first_pid = 1

test_mask = (pids == first_pid)
train_mask = ~test_mask

X_tr = X[train_mask]
y_tr = y[train_mask]
train_pids = pids[train_mask]

X_te = X[test_mask]
y_te = y[test_mask]

print(f"Test fold: holding out p{first_pid}")
print(f"  train windows: {len(y_tr)}, test windows: {len(y_te)}")

preds, probs, info = train_one_fold(
    X_tr,
    y_tr,
    train_pids,
    X_te,
    y_te,
    first_pid
)

acc = accuracy_score(y_te, preds)
bacc = balanced_accuracy_score(y_te, preds)
f1 = f1_score(y_te, preds, zero_division=0)

print(f"\n  Best epoch        : {info['best_epoch']}")
print(f"  Best val loss     : {info['best_val_loss']:.4f}")
print(f"  Validation pids   : {info['val_pids']}")

print(f"\n  Accuracy          : {acc:.3f}")
print(f"  Balanced accuracy : {bacc:.3f}")
print(f"  F1 (stress)       : {f1:.3f}")
print(f"  Confusion matrix  :\n{confusion_matrix(y_te, preds)}")

In [ ]:
results = []

for X_tr, y_tr, X_te, y_te, held_out in loso_splits(
    X, y, pids, verbose=False
):
    train_pids = pids[pids != held_out]

    preds, probs, info = train_one_fold(
        X_tr,
        y_tr,
        train_pids,
        X_te,
        y_te,
        held_out
    )

    acc = accuracy_score(y_te, preds)
    bacc = balanced_accuracy_score(y_te, preds)
    f1 = f1_score(y_te, preds, zero_division=0)

    results.append({
        "participant": held_out,
        "accuracy": acc,
        "balanced_acc": bacc,
        "f1": f1,
        "best_epoch": info["best_epoch"],
        "best_val_loss": info["best_val_loss"],
        "n_test": len(y_te)
    })

    print(
        f"p{held_out:2d} — "
        f"acc={acc:.3f}  "
        f"bacc={bacc:.3f}  "
        f"f1={f1:.3f}  "
        f"best_epoch={info['best_epoch']}"
    )

accs = [r["accuracy"] for r in results]
baccs = [r["balanced_acc"] for r in results]
f1s = [r["f1"] for r in results]
best_epochs = [r["best_epoch"] for r in results]

print("\n" + "=" * 56)
print("BiLSTM + STATS + EARLY STOPPING — LOSO RESULTS")
print("=" * 56)
print(f"Participants evaluated : {len(results)}")
print(f"Mean accuracy          : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Mean balanced accuracy : {np.mean(baccs):.3f} ± {np.std(baccs):.3f}")
print(f"Mean F1 (stress)       : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
print(f"Mean best epoch        : {np.mean(best_epochs):.1f}")
print("=" * 56)